In [1]:
import yfinance as yf

In [4]:
sp500 = yf.Ticker("^GSPC")

In [5]:
sp500 = sp500.history(period="max")

In [6]:
sp500

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
1927-12-30 00:00:00-05:00,17.660000,17.660000,17.660000,17.660000,0,0.0,0.0
1928-01-03 00:00:00-05:00,17.760000,17.760000,17.760000,17.760000,0,0.0,0.0
1928-01-04 00:00:00-05:00,17.719999,17.719999,17.719999,17.719999,0,0.0,0.0
1928-01-05 00:00:00-05:00,17.549999,17.549999,17.549999,17.549999,0,0.0,0.0
1928-01-06 00:00:00-05:00,17.660000,17.660000,17.660000,17.660000,0,0.0,0.0
...,...,...,...,...,...,...,...
2025-01-03 00:00:00-05:00,5891.069824,5949.339844,5888.660156,5942.470215,3667340000,0.0,0.0
2025-01-06 00:00:00-05:00,5982.810059,6021.040039,5960.009766,5975.379883,4940120000,0.0,0.0
2025-01-07 00:00:00-05:00,5993.259766,6000.680176,5890.680176,5909.029785,4517330000,0.0,0.0


In [8]:
del sp500["Dividends"]
del sp500["Stock Splits"]

In [9]:
sp500["Tomorrow"] = sp500["Close"].shift(-1)

In [10]:
sp500

,Open,High,Low,Close,Volume,Tomorrow
Date,,,,,,
1927-12-30 00:00:00-05:00,17.660000,17.660000,17.660000,17.660000,0,17.760000
1928-01-03 00:00:00-05:00,17.760000,17.760000,17.760000,17.760000,0,17.719999
1928-01-04 00:00:00-05:00,17.719999,17.719999,17.719999,17.719999,0,17.549999
1928-01-05 00:00:00-05:00,17.549999,17.549999,17.549999,17.549999,0,17.660000
1928-01-06 00:00:00-05:00,17.660000,17.660000,17.660000,17.660000,0,17.500000
...,...,...,...,...,...,...
2025-01-03 00:00:00-05:00,5891.069824,5949.339844,5888.660156,5942.470215,3667340000,5975.379883
2025-01-06 00:00:00-05:00,5982.810059,6021.040039,5960.009766,5975.379883,4940120000,5909.029785
2025-01-07 00:00:00-05:00,5993.259766,6000.680176,5890.680176,5909.029785,4517330000,5918.250000


In [11]:
sp500["Target"] = (sp500["Tomorrow"] > sp500["Close"]).astype(int)

In [12]:
sp500

,Open,High,Low,Close,Volume,Tomorrow,Target
Date,,,,,,,
1927-12-30 00:00:00-05:00,17.660000,17.660000,17.660000,17.660000,0,17.760000,1
1928-01-03 00:00:00-05:00,17.760000,17.760000,17.760000,17.760000,0,17.719999,0
1928-01-04 00:00:00-05:00,17.719999,17.719999,17.719999,17.719999,0,17.549999,0
1928-01-05 00:00:00-05:00,17.549999,17.549999,17.549999,17.549999,0,17.660000,1
1928-01-06 00:00:00-05:00,17.660000,17.660000,17.660000,17.660000,0,17.500000,0
...,...,...,...,...,...,...,...
2025-01-03 00:00:00-05:00,5891.069824,5949.339844,5888.660156,5942.470215,3667340000,5975.379883,1
2025-01-06 00:00:00-05:00,5982.810059,6021.040039,5960.009766,5975.379883,4940120000,5909.029785,0
2025-01-07 00:00:00-05:00,5993.259766,6000.680176,5890.680176,5909.029785,4517330000,5918.250000,1


In [13]:
sp500 = sp500.loc["1990-01-01":].copy()

In [14]:
sp500

,Open,High,Low,Close,Volume,Tomorrow,Target
Date,,,,,,,
1990-01-02 00:00:00-05:00,353.399994,359.690002,351.980011,359.690002,162070000,358.760010,0
1990-01-03 00:00:00-05:00,359.690002,360.589996,357.890015,358.760010,192330000,355.670013,0
1990-01-04 00:00:00-05:00,358.760010,358.760010,352.890015,355.670013,177000000,352.200012,0
1990-01-05 00:00:00-05:00,355.670013,355.670013,351.350006,352.200012,158530000,353.790009,1
1990-01-08 00:00:00-05:00,352.200012,354.239990,350.540009,353.790009,140110000,349.619995,0
...,...,...,...,...,...,...,...
2025-01-03 00:00:00-05:00,5891.069824,5949.339844,5888.660156,5942.470215,3667340000,5975.379883,1
2025-01-06 00:00:00-05:00,5982.810059,6021.040039,5960.009766,5975.379883,4940120000,5909.029785,0
2025-01-07 00:00:00-05:00,5993.259766,6000.680176,5890.680176,5909.029785,4517330000,5918.250000,1


In [18]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, min_samples_split=100, random_state=1)

train = sp500.iloc[:-100]
test = sp500.iloc[-100:]

predictors = ["Close", "Volume", "Open", "High", "Low"]
model.fit(train[predictors], train["Target"])

RandomForestClassifier(min_samples_split=100, random_state=1)

In [24]:
from sklearn.metrics import precision_score

preds = model.predict(test[predictors])

In [25]:
import pandas as pd

preds = pd.Series(preds, index=test.index)

In [26]:
precision_score(test["Target"], preds)

0.5205479452054794

In [27]:
combined = pd.concat([test["Target"], preds], axis=1)

In [28]:
def predict(train, test, predictors, model):
    model.fit(train[predictors], train["Target"])
    preds = model.predict(test[predictors])
    preds = pd.Series(preds, index=test.index, name="Predictions")
    combined = pd.concat([test["Target"], preds], axis=1)
    return combined

In [33]:
def backtest(data, model, predictors, start=2500, step=250):
    all_predictors =[]

    for i in range(start, data.shape[0], step):
        train = data.iloc[0:i].copy()
        test = data.iloc[i:(i+step)].copy()
        predictions = predict(train, test, predictors, model)
        all_predictors.append(predictions)
    return pd.concat(all_predictors)

In [34]:
predictions = backtest(sp500, model, predictors)

In [36]:
predictions["Predictions"].value_counts()

Predictions
0    3687
1    2636
Name: count, dtype: int64

In [38]:
precision_score(predictions["Target"], predictions["Predictions"])

0.5284522003034902

In [40]:
predictions["Target"].value_counts() / predictions.shape[0]

Target
1    0.535822
0    0.464178
Name: count, dtype: float64

In [54]:
horizons = [2,5,60,250,1000]
new_predictors = []

for horizon in horizons:
    rolling_averages = sp500.rolling(horizon).mean()

    ratio_column = f"Close_Ratio_{horizon}"
    sp500[ratio_column] = sp500["Close"] / rolling_averages["Close"]

    trend_column = f"Trend_{horizons}"
    sp500[trend_column] = sp500.shift(1).rolling(horizon).sum()["Target"]

    new_predictors += [ratio_column, trend_column] 

C:\Users\sujay\AppData\Local\Temp\ipykernel_20000\4179637693.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sp500[ratio_column] = sp500["Close"] / rolling_averages["Close"]
C:\Users\sujay\AppData\Local\Temp\ipykernel_20000\4179637693.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sp500[trend_column] = sp500.shift(1).rolling(horizon).sum()["Target"]


In [55]:
sp500 = sp500.dropna()

In [56]:
sp500

,Open,High,Low,Close,Volume,Tomorrow,Target,Close_Ratio_2,"trend_[2, 5, 60, 250, 1000]",Close_Ratio_5,Close_Ratio_60,Close_Ratio_250,Close_Ratio_1000,"Trend_[2, 5, 60, 250, 1000]"
Date,,,,,,,,,,,,,,
1997-11-26 00:00:00-05:00,950.820007,956.469971,950.820007,951.640015,487750000,955.400024,1,1.000431,563.0,0.997275,1.009111,1.113282,1.517046,563.0
1997-11-28 00:00:00-05:00,951.640015,959.130005,951.640015,955.400024,189070000,974.770020,1,1.001972,564.0,1.001967,1.012659,1.116599,1.521842,564.0
1997-12-01 00:00:00-05:00,955.400024,974.770020,955.400024,974.770020,590300000,971.679993,0,1.010035,564.0,1.019783,1.032356,1.138015,1.551433,564.0
1997-12-02 00:00:00-05:00,974.780029,976.200012,969.830017,971.679993,576120000,976.770020,1,0.998412,563.0,1.011259,1.028349,1.133204,1.545272,563.0
1997-12-03 00:00:00-05:00,971.679993,980.809998,966.159973,976.770020,624610000,973.099976,0,1.002612,564.0,1.011095,1.032949,1.137882,1.552105,564.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-01-02 00:00:00-05:00,5903.259766,5935.089844,5829.529785,5868.549805,3621680000,5942.470215,1,0.998887,527.0,0.989119,0.990879,1.079057,1.296134,527.0
2025-01-03 00:00:00-05:00,5891.069824,5949.339844,5888.660156,5942.470215,3667340000,5975.379883,1,1.006259,527.0,1.004799,1.002820,1.091649,1.311840,527.0
2025-01-06 00:00:00-05:00,5982.810059,6021.040039,5960.009766,5975.379883,4940120000,5909.029785,0,1.002761,527.0,1.010209,1.007854,1.096718,1.318475,527.0


In [59]:
model = RandomForestClassifier(n_estimators=200, min_samples_split=50, random_state=1)

In [62]:
def predict(train, test, predictors, model):
    model.fit(train[predictors], train["Target"])
    preds = model.predict_proba(test[predictors])[:,1]
    preds[preds >= .6] = 1
    preds[preds < .6] = 0
    preds = pd.Series(preds, index=test.index, name="Predictions")
    combined = pd.concat([test["Target"], preds], axis=1)
    return combined

In [63]:
predictions = backtest(sp500, model, new_predictors)

In [69]:
predictions["Predictions"].value_counts()

Predictions
0.0    3678
1.0     644
Name: count, dtype: int64

In [70]:
precision_score(predictions["Target"], predictions["Predictions"])

0.5232919254658385

In [67]:
predictions["Target"].value_counts() / predictions.shape[0]

Target
1    0.542342
0    0.457658
Name: count, dtype: float64